# The instrument decided, not the model

**RSNA Knee Abnormality Detection** — final public leaderboard 0.903 macro AUC,
rank ~4,000/6,000. The rank looks worse than the work: a large share of the
entries above this one are running shared/borrowed pretrained weights rather
than models trained on the competition data (see the credits section of the
[full repo](https://github.com/abhirick23/rsna-knee-abnormality-detection)).

The score isn't the finding worth publishing. The finding is this: **the
only expert ground truth available in this competition — 58 labeled
studies, out of 4,407 — has a standard error on the competition metric
(macro AUC) of roughly 0.03, and worse per individual finding (~0.09 for a
rare label like Lateral Meniscus, which has only ~10 positive cases in the
gold set).** That's coarser than most of the differences this project
needed to detect between model variants. Several "promising" results during
training were, on closer inspection, statistically indistinguishable from
noise on that gold set.

This notebook reproduces the two pieces of evidence for that claim —
one simulated, one real — with the bootstrap code and the figure both
running live below, not pasted in as static output.

## Panel A (simulated): the resolution cliff

Inject a *known* true effect — a +0.014 macro-AUC improvement — into
synthetic data shaped like this competition (4,407 studies, 12 binary
findings, ~25% prevalence), and measure it with **paired bootstrap** at
three sample sizes: the 58-study gold set, one held-out fold (870 studies),
and out-of-fold across all 4,349 report-labeled studies.

Paired bootstrap resamples the *same* studies for both models being
compared each time, which cancels most of the shared variance and resolves
roughly 3-4x finer than comparing two independent point estimates — but
even that isn't enough at n=58.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]


def _binarise(T):
    return np.asarray(T) >= 0.5


def per_label_auc(P, T):
    T = _binarise(T)
    out = []
    for k in range(T.shape[1]):
        pos = T[:, k].sum()
        out.append(roc_auc_score(T[:, k], P[:, k]) if 0 < pos < len(T) else np.nan)
    return out


def macro_auc(P, T):
    return float(np.nanmean(per_label_auc(P, T)))


def paired_bootstrap(Pa, Pb, T, n=2000, seed=0, alpha=0.05):
    '''CI for macro_auc(A) - macro_auc(B), both scored on the same resample.

    A CI containing 0 means the data cannot distinguish the two models,
    whatever the point estimate says.
    '''
    rng = np.random.RandomState(seed)
    N = len(T)
    diffs = []
    for _ in range(n):
        i = rng.randint(0, N, N)
        Ti = _binarise(T[i])
        if Ti.sum(0).min() == 0 or Ti.sum(0).max() == N:
            continue
        diffs.append(macro_auc(Pa[i], T[i]) - macro_auc(Pb[i], T[i]))
    diffs = np.asarray(diffs)
    return (float(diffs.mean()),
            float(np.percentile(diffs, 100 * alpha / 2)),
            float(np.percentile(diffs, 100 * (1 - alpha / 2))))


def resolution_cliff(seed=0):
    rng = np.random.RandomState(seed)
    out = []
    for N in (58, 870, 4349):
        T = (rng.rand(N, 12) < 0.25).astype(float)
        signal = T * 1.0
        base = signal + rng.randn(N, 12) * 1.00
        better = signal + rng.randn(N, 12) * 0.92   # ~0.01-0.02 macro AUC better
        out.append((N,) + paired_bootstrap(better, base, T, n=500, seed=seed))
    return out


for N, m, lo, hi in resolution_cliff():
    resolvable = lo > 0
    print(f"n={N:5d}  diff {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  "
          f"{'resolvable' if resolvable else 'NOT resolvable'}  "
          f"(CI width {hi - lo:.3f})")


The same true +0.014 effect: unresolvable at n=58, unresolvable (borderline)
at n=870, clearly resolvable at n=4,349. The sample size decided the verdict,
not the model.

## Panel B (real, measured): the decision that nearly got made on noise

This is not simulated. It's the Lateral Meniscus AUC, evaluated against the
58 gold studies, from one real training run (a 5-plane variant), read once
per epoch:

```
epoch:  0      1      2      3      4      5      6
AUC:   .748   .805   .817   .706   .752   .740   .725
```

At epoch 2, 0.817 was briefly read as confirming a hypothesis about which
MRI sequences (fluid-sensitive vs. PD/T1) the model needed to see menisci
well. One epoch later it was 0.706. Lateral Meniscus has about 10 positive
cases in 58 gold studies, giving a per-label standard error near 0.09 — the
full epoch curve is noise around that scale, not a trend, and the 0.817
reading was never replicated.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

THEME = dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781",
             grid="#e1e0d9", axis="#c3c2b7", series="#2a78d6", critical="#d03b3b")
c = THEME

LATMEN = [0.748, 0.805, 0.817, 0.706, 0.752, 0.740, 0.725]
SE = 0.09  # ~10 positives in 58 studies

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"],
    "text.color": c["ink"], "axes.labelcolor": c["ink2"],
    "xtick.color": c["muted"], "ytick.color": c["muted"],
    "axes.edgecolor": c["axis"],
})
fig, (axA, axB) = plt.subplots(
    1, 2, figsize=(12.0, 5.4), facecolor=c["surface"],
    gridspec_kw=dict(wspace=0.30, left=0.095, right=0.97, top=0.78, bottom=0.135))

fig.text(0.095, 0.945, "The instrument decided, not the model",
         fontsize=16, fontweight="bold", color=c["ink"], ha="left", va="top")

# Panel A
rows = resolution_cliff()
ys = np.arange(len(rows))[::-1]
axA.set_facecolor(c["surface"])
axA.axvline(0, color=c["axis"], lw=1.2, zorder=1)
for y, (N, m, lo, hi) in zip(ys, rows):
    resolvable = lo > 0
    col = c["series"] if resolvable else c["critical"]
    axA.plot([lo, hi], [y, y], lw=2, color=col, solid_capstyle="round", zorder=3)
    axA.plot([m], [y], "o", ms=9, color=col, mec=c["surface"], mew=2, zorder=4)
    label = "resolves the effect" if resolvable else "cannot resolve the effect"
    axA.text(hi + 0.005, y + 0.22, label, va="bottom", fontsize=10,
             color=c["ink"] if resolvable else c["critical"],
             fontweight="bold" if resolvable else "normal")
    axA.text(hi + 0.005, y - 0.22, f"CI width {hi - lo:.3f}", va="top",
             fontsize=9, color=c["muted"])
axA.set_yticks(ys)
axA.set_yticklabels([f"n = {N:,}\n{lab}" for (N, *_), lab in
                     zip(rows, ["gold set", "one fold", "out-of-fold"])],
                    fontsize=10.5, color=c["ink"])
axA.set_xlabel("measured difference in macro AUC  (95% CI)", fontsize=10)
axA.set_xlim(-0.075, 0.125)
axA.set_ylim(-0.6, 2.6)
axA.xaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:+.02f}"))
axA.grid(axis="x", color=c["grid"], lw=0.8, zorder=0)
axA.set_axisbelow(True)
for s in ("top", "right", "left"):
    axA.spines[s].set_visible(False)
axA.tick_params(axis="y", length=0)
axA.set_title("A. One true +0.014 improvement, three sample sizes (simulated)",
              fontsize=11, color=c["ink"], loc="left", pad=12)

# Panel B
x = np.arange(len(LATMEN))
axB.set_facecolor(c["surface"])
axB.fill_between(x, np.array(LATMEN) - SE, np.array(LATMEN) + SE,
                 color=c["series"], alpha=0.13, lw=0, zorder=1)
axB.plot(x, LATMEN, lw=2, color=c["series"], solid_capstyle="round", zorder=3)
axB.plot(x, LATMEN, "o", ms=7, color=c["series"], mec=c["surface"], mew=2, zorder=4)
axB.plot([2], [LATMEN[2]], "o", ms=11, color=c["critical"], mec=c["surface"], mew=2, zorder=5)
axB.annotate("0.817 -- read at the time\nas the hypothesis confirmed",
             xy=(2, LATMEN[2]), xytext=(2.45, 0.878), fontsize=9.5, color=c["critical"],
             va="top", arrowprops=dict(arrowstyle="-", color=c["critical"], lw=1.2))
axB.annotate("0.706, one epoch later", xy=(3, LATMEN[3]), xytext=(3.3, 0.655),
             fontsize=9.5, color=c["ink2"], va="top",
             arrowprops=dict(arrowstyle="-", color=c["muted"], lw=1.2))
axB.set_xticks(x); axB.set_xticklabels([f"ep{i}" for i in x], fontsize=10)
axB.set_ylim(0.60, 0.94)
axB.set_ylabel("Lateral Meniscus AUC", fontsize=10)
axB.grid(axis="y", color=c["grid"], lw=0.8, zorder=0)
axB.set_axisbelow(True)
for s in ("top", "right"):
    axB.spines[s].set_visible(False)
axB.set_title("B. One model, scored each epoch on 58 gold studies (measured)",
              fontsize=11, color=c["ink"], loc="left", pad=12)
axB.text(0.04, 0.617, "shaded band = +/-1 SE for ~10 positives", fontsize=9, color=c["muted"])

plt.show()


## What this means for reading any result on a small labeled set

- A result landing inside an instrument's noise band means the instrument
  couldn't resolve it — not that the true effect is zero. Several ablations
  in the full project (backbone size, slice count, extra imaging planes)
  were "null" on the 58-study gold set in exactly this sense.
- The clearest demonstration in the full project: a 3-plane vs. 5-plane
  model pair, same folds, same labels, differed by **0.0177** on the
  58-study gold set and **0.0001** out-of-fold across the 4,349
  report-labeled studies. Same models — the instrument decided which
  answer looked true.
- Practical takeaway: prefer paired comparisons over point estimates,
  validate on the largest available instrument (out-of-fold, not a small
  expert-labeled holdout) before trusting a result, and state null results
  as "this instrument couldn't resolve it" rather than "this doesn't help."

## Full pipeline and writeup

This notebook is the narrative version of one finding. The full
preprocessing, labeling, training, ablation table (11 experiments, 1 kept),
and limitations/future-work are here:

**[github.com/abhirick23/rsna-knee-abnormality-detection](https://github.com/abhirick23/rsna-knee-abnormality-detection)**

The report-label dataset used alongside the images in this project is
published separately: [RSNA Knee Report Labels (Gemini)](https://www.kaggle.com/datasets/abhiroopsarkar/rsna-knee-gemini-report-labels).

Code in this notebook is released under the MIT license, matching the
GitHub repo.